In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def visualize_heartbeat(hub_name, trains, bikes, config,
                       zoom_start, zoom_end, event_type='arrival'):
    """
    Plots a segment of time showing Train Events, Pressure Signal, and Bike Actuals.
    """
    print(f"Generating Heartbeat Plot for {hub_name} ({zoom_start} to {zoom_end})...")

    # 1. Filter Data to Zoom Window
    t_mask = (trains[f'actual_{event_type}'] >= zoom_start) & (trains[f'actual_{event_type}'] <= zoom_end)
    zoom_trains = trains[t_mask].copy()

    b_start = pd.to_datetime(zoom_start)
    b_end = pd.to_datetime(zoom_end)

    # 2. Generate the Theoretical Signal (The "Pressure")
    timeline = pd.date_range(b_start, b_end, freq='1min')
    sig_arr, sig_dep = generate_signals(zoom_trains, timeline, config['params'])

    # Select signal based on event type
    pressure_signal = sig_arr if event_type == 'arrival' else sig_dep
    bike_col = 'start_time' if event_type == 'arrival' else 'end_time'

    # 3. Get Actual Bike Counts
    # Filter bikes in this window
    zoom_bikes = bikes[(bikes[bike_col] >= b_start) & (bikes[bike_col] <= b_end)]
    bike_counts = zoom_bikes.set_index(bike_col).resample('1min').size().reindex(timeline, fill_value=0)

    # --- PLOTTING ---
    fig, ax1 = plt.subplots(figsize=(15, 7))

    # A. Plot Bike Rentals (Bars)
    ax1.bar(timeline, bike_counts, width=0.0007, color='#004c6d', alpha=0.6, label='Actual Bike Rentals')
    ax1.set_ylabel('Number of Bikes', color='#004c6d', fontsize=12)
    ax1.tick_params(axis='y', labelcolor='#004c6d')
    ax1.set_ylim(0, bike_counts.max() * 1.5) # Scale up to leave room for events

    # B. Plot Pressure Signal (Smooth Curve)
    ax2 = ax1.twinx()
    color_sig = '#d55e00' if event_type == 'arrival' else '#009e73'
    ax2.plot(timeline, pressure_signal, color=color_sig, linewidth=2, label='Passenger Pressure (Model)')
    ax2.fill_between(timeline, pressure_signal, color=color_sig, alpha=0.1)
    ax2.set_ylabel('Theoretical Pressure', color=color_sig, fontsize=12)
    ax2.tick_params(axis='y', labelcolor=color_sig)

    # C. Plot Individual Train Events (Markers)
    # We plot them at y=0 on the secondary axis, but with tall vertical lines
    y_min, y_max = ax2.get_ylim()

    for _, t in zoom_trains.iterrows():
        t_time = t[f'actual_{event_type}']
        t_cat = t.get('train_type', '?')

        # Color code markers
        marker_color = 'red' if t_cat in ['Os', 'Sp'] else 'gray'
        alpha_line = 0.8 if t_cat in ['Os', 'Sp'] else 0.4

        ax2.vlines(x=t_time, ymin=0, ymax=y_max*0.2, color=marker_color, linestyle='--', alpha=alpha_line)
        ax2.text(t_time, y_max*0.22, t_cat, ha='center', fontsize=8, color=marker_color, rotation=90)

    # Formatting
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    plt.title(f"The Heartbeat of {hub_name}: {event_type.title()}s\nWindow: {zoom_start} to {zoom_end}", fontsize=14)
    plt.grid(True, alpha=0.3)

    # Custom Legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

    plt.show()

# --- EXAMPLE USAGE ---
# Assuming you have loaded 't', 'b', and 'NETWORKS' from previous code
# visualize_heartbeat('Ostrava_Svinov', t, b, NETWORKS['Ostrava_Svinov'],
#                     '2023-06-01 06:00', '2023-06-01 10:00', event_type='arrival')

In [ ]:
def visualize_results(results_df):
    """
    Takes the results DataFrame from run_master_analysis() and plots the comparison.
    """
    # Filter for just one metric (e.g., Arrival Correlation) to keep it clean
    df_plot = results_df[results_df['Category'].isin(['Regional', 'LongDist'])].copy()

    plt.figure(figsize=(12, 6))

    # Grouped Bar Chart
    sns.barplot(data=df_plot, x='Hub', y='Arr_Corr', hue='Category', palette='viridis')

    plt.axhline(0, color='black', linewidth=1)
    plt.title("Hypothesis Check: Do Regional Trains Drive More Demand?", fontsize=14)
    plt.ylabel("Correlation Strength (Pearson r)", fontsize=12)
    plt.xlabel("City / Hub", fontsize=12)
    plt.grid(axis='y', alpha=0.3)
    plt.legend(title='Train Type')

    plt.show()